<a href="https://colab.research.google.com/github/mkobycheva/recommendation-system/blob/a-b-test/notebooks/domain_ablation_a_b_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-domain vs multi-domain: A/B test

Same LLM-agent methodology as `a_b_test.ipynb` (ALS vs BERT), but the two
conditions being compared are single-domain-trained vs multi-domain
(collective/joint) versions of the *same* model -- done once for ALS, once
for BERT4Rec. Unlike `a_b_test.ipynb`, this is a **paired** design: each
sampled user gets recommendations from both conditions, so the comparison
is a one-sample z-test on the per-user (multi - single) rate difference,
not two independent samples.

In [9]:
!test -d /content/recommendation-system/.git \
  || git clone -b a-b-test https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git checkout a-b-test
!git pull origin a-b-test

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

/content/recommendation-system
Already on 'a-b-test'
Your branch is up to date with 'origin/a-b-test'.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 643 bytes | 321.00 KiB/s, done.
From https://github.com/mkobycheva/recommendation-system
 * branch            a-b-test   -> FETCH_HEAD
   0d4ee41..fbabfbe  a-b-test   -> origin/a-b-test
Updating 0d4ee41..fbabfbe
Fast-forward
 notebooks/domain_ablation_a_b_test.ipynb | 5 +++--
 1 file changed, 3 insertions(+), 2 deletions(-)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip install -q anthropic

In [11]:
import os
import numpy as np
import pandas as pd
import json
import time
import random
import random as pyrandom
from concurrent.futures import ThreadPoolExecutor, as_completed
from src.data.build_matrix import add_domain_item_ids
from anthropic import Anthropic, RateLimitError, APIConnectionError, InternalServerError, APITimeoutError

In [12]:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [13]:
key = os.environ.get("ANTHROPIC_API_KEY")
print(f"key present: {key is not None}, length: {len(key) if key else 0}, starts with: {key[:7] if key else None}")

key present: True, length: 73, starts with: sk-or-v


## Getting the data

In [14]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'
AB_TEST_DIR = f'{DATA_DIR}/a-b-test'

books_train = add_domain_item_ids(pd.read_csv(f'{DATA_DIR}/books_train.csv'), 'books')
movies_train = add_domain_item_ids(pd.read_csv(f'{DATA_DIR}/movies_train.csv'), 'movies')

books_meta = pd.read_csv(f'{DATA_DIR}/books_meta.csv')
movies_meta = pd.read_csv(f'{DATA_DIR}/movies_meta.csv')
books_title = books_meta.set_index('parent_asin')['title'].to_dict()
movies_title = movies_meta.set_index('parent_asin')['title'].to_dict()


def title_for(item_id):
    domain, asin = item_id.split('::', 1)
    lookup = books_title if domain == 'books' else movies_title
    return lookup.get(asin, '(title not found)')

## Shared helpers (persona generation, LLM evaluation)

Unchanged from `a_b_test.ipynb` -- same structured-output personas and
funnel-logic engagement predictions, just reused here for a different pair
of conditions.

In [15]:
client = Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url="https://openrouter.ai/api",
)

PROFILE_MODEL = "anthropic/claude-haiku-4.5"   # OpenRouter slug
EVAL_MODEL = "anthropic/claude-haiku-4.5"
MAX_WORKERS = 8
MAX_RETRIES = 5
RETRYABLE_ERRORS = (RateLimitError, APIConnectionError, InternalServerError, APITimeoutError)

MIN_HISTORY_ITEMS = 3   # need at least this many valid historical items to build a profile
MAX_HISTORY_ITEMS = 20  # cap how much history we send to the agent (cost/context control)
N_USERS_PER_DOMAIN = 250

PROFILE_TOOL = {
    "name": "submit_user_profile",
    "description": "Submit a synthetic user profile inferred from their rating history.",
    "input_schema": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "2-3 sentence persona summary of this reader/viewer."
            },
            "age": {"type": "integer", "description": "Plausible age estimate, 18-80."},
            "interests": {
                "type": "array",
                "items": {"type": "string"},
                "description": "3-6 general interests/hobbies suggested by their history."
            },
            "favorite_genres": {
                "type": "array",
                "items": {"type": "string"},
                "description": "2-5 genres/categories this person clearly favors."
            },
        },
        "required": ["description", "age", "interests", "favorite_genres"],
    },
}

PROFILE_SYSTEM = (
    "You infer realistic reader/viewer personas from a user's rating history. "
    "Base your inference only on the titles and ratings given. Be specific and concrete, "
    "not generic. Only items the user rated highly reflect genuine preference."
)

EVAL_TOOL = {
    "name": "submit_recommendation_evaluation",
    "description": "Submit engagement predictions for each recommended item, from this user's perspective.",
    "input_schema": {
        "type": "object",
        "properties": {
            "evaluations": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "item_id": {"type": "string", "description": "Echo back the exact item_id given."},
                        "will_view": {"type": "boolean", "description": "Would click into/view this item at all."},
                        "will_add_to_cart": {"type": "boolean", "description": "Would add to cart. Can only be true if will_view is true."},
                        "will_buy": {"type": "boolean", "description": "Would actually purchase. Can only be true if will_add_to_cart is true."},
                        "reason": {"type": "string", "description": "One short phrase, <=15 words."},
                    },
                    "required": ["item_id", "will_view", "will_add_to_cart", "will_buy", "reason"],
                },
            },
        },
        "required": ["evaluations"],
    },
}

EVAL_SYSTEM = (
    "You role-play as a specific person, using their profile and rating history, to predict "
    "realistic e-commerce behavior toward a list of recommended items. For each item, decide "
    "whether this specific person would view it, add it to cart, and buy it. "
    "These are a funnel: will_add_to_cart can only be true if will_view is true, and "
    "will_buy can only be true if will_add_to_cart is true. Most items should NOT be bought - "
    "be realistic and selective, the way a real shopper is."
)


def call_claude_tool(model, system, user_message, tool, max_retries=MAX_RETRIES):
    """Call Claude, forcing a structured tool-call response. Retries only on transient errors."""
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=model,
                max_tokens=1024,
                system=system,
                messages=[{"role": "user", "content": user_message}],
                tools=[tool],
                tool_choice={"type": "tool", "name": tool["name"]},
            )
            for block in resp.content:
                if block.type == "tool_use":
                    return block.input
            raise ValueError(f"No tool_use block returned: {resp.content}")
        except RETRYABLE_ERRORS as e:
            last_err = e
            time.sleep(min(2 ** attempt + pyrandom.random(), 30))
        except Exception as e:
            raise RuntimeError(f"Non-retryable error: {type(e).__name__}: {e}") from e

    raise RuntimeError(f"Failed after {max_retries} retries: {type(last_err).__name__}: {last_err}") from last_err


def has_valid_recs(entry):
    """True if a recs_bulk entry has clean, non-null recommendation data."""
    ids = entry.get('recommended_item_ids')
    titles = entry.get('recommended_titles')
    if not ids or not titles or len(ids) != len(titles):
        return False
    if any(pd.isna(x) or x in (None, '') for x in ids):
        return False
    if any(pd.isna(x) or x in (None, '') for x in titles):
        return False
    return True


def get_valid_history(group, min_items=MIN_HISTORY_ITEMS, max_items=MAX_HISTORY_ITEMS):
    """Most-recent-first (title, rating) history for one user's train rows."""
    rows = group.dropna(subset=['item_id', 'rating']).sort_values('timestamp', ascending=False)

    history = []
    for _, r in rows.iterrows():
        title = title_for(r['item_id'])
        if title == '(title not found)':
            continue
        history.append({'title': title, 'rating': float(r['rating'])})
        if len(history) >= max_items:
            break

    return history if len(history) >= min_items else None


def build_profile_prompt(user_entry):
    domain = user_entry["domain"]
    lines = [f"- \"{h['title']}\" (rated {h['rating']}/5)" for h in user_entry["history"]]
    return (
        f"Here is a user's {domain} rating history, most recent first:\n"
        + "\n".join(lines)
        + f"\n\nInfer a plausible persona for this {domain} reader/viewer."
    )


def generate_profile(user_entry):
    prompt = build_profile_prompt(user_entry)
    profile = call_claude_tool(PROFILE_MODEL, PROFILE_SYSTEM, prompt, PROFILE_TOOL)
    return {**user_entry, "profile": profile}


def generate_profiles(sampled_users, save_path, max_workers=MAX_WORKERS, checkpoint_every=10):
    results = {}
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f"Resuming: {len(results)} profiles already saved at {save_path}")

    todo = [u for u in sampled_users if u["user_id"] not in results]
    print(f"{len(todo)} profiles to generate ({len(sampled_users) - len(todo)} already done)")

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(generate_profile, u): u for u in todo}
        done, failed = 0, 0
        start = time.time()
        for future in as_completed(futures):
            user_entry = futures[future]
            try:
                result = future.result()
                results[result["user_id"]] = result
            except Exception as e:
                failed += 1
                print(f"FAILED user_id={user_entry['user_id']}: {type(e).__name__}: {e}")
            done += 1
            if done % checkpoint_every == 0 or done == len(todo):
                with open(save_path, "w") as f:
                    json.dump(results, f, indent=2)
                elapsed = time.time() - start
                print(f"  {done}/{len(todo)} processed ({failed} failed), {elapsed:.0f}s elapsed, checkpointed")

    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    return results


def build_eval_prompt(record, shuffled_items):
    domain = record["domain"]
    p = record["profile"]
    history_lines = "\n".join(f"- \"{h['title']}\" (rated {h['rating']}/5)" for h in record["history"])
    items_lines = "\n".join(f"- item_id: {it['item_id']} | title: \"{it['title']}\"" for it in shuffled_items)

    return f"""User profile:
- Description: {p['description']}
- Age: {p['age']}
- Interests: {', '.join(p['interests'])}
- Favorite genres: {', '.join(p['favorite_genres'])}

This user's {domain} rating history:
{history_lines}

Here are 10 recommended {domain} items shown to this user (in random order, not ranked):
{items_lines}

For each item, predict whether this specific user would view it, add it to cart, and buy it."""


def evaluate_user_condition(record, condition):
    """condition is 'multi' or 'single' -- picks which of the two paired
    recs lists on `record` to evaluate (see sample_paired_users below)."""
    item_ids = record[f'{condition}_recommended_item_ids']
    titles = record[f'{condition}_recommended_titles']
    items = [
        {"item_id": iid, "title": title, "rank": rank + 1}
        for rank, (iid, title) in enumerate(zip(item_ids, titles))
    ]
    shuffled = items.copy()
    pyrandom.shuffle(shuffled)

    prompt = build_eval_prompt(record, shuffled)
    output = call_claude_tool(EVAL_MODEL, EVAL_SYSTEM, prompt, EVAL_TOOL)

    rank_by_id = {it["item_id"]: it["rank"] for it in items}
    evaluations = []
    for e in output["evaluations"]:
        # Safety net: enforce funnel logic in code too, in case the model slips
        will_view = bool(e["will_view"])
        will_cart = bool(e["will_add_to_cart"]) and will_view
        will_buy = bool(e["will_buy"]) and will_cart
        evaluations.append({
            "item_id": e["item_id"],
            "rank": rank_by_id.get(e["item_id"]),
            "will_view": will_view,
            "will_add_to_cart": will_cart,
            "will_buy": will_buy,
            "reason": e.get("reason", ""),
        })

    return {
        "user_id": record["user_id"],
        "domain": record["domain"],
        "condition": condition,
        "evaluations": evaluations,
        "n_view": sum(e["will_view"] for e in evaluations),
        "n_cart": sum(e["will_add_to_cart"] for e in evaluations),
        "n_buy": sum(e["will_buy"] for e in evaluations),
    }


def evaluate_users(records, condition, save_path, max_workers=MAX_WORKERS, checkpoint_every=10):
    results = {}
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f"Resuming: {len(results)} '{condition}' evaluations already saved at {save_path}")

    todo = [r for r in records if r["user_id"] not in results]
    print(f"{len(todo)} '{condition}' evaluations to run ({len(records) - len(todo)} already done)")

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(evaluate_user_condition, r, condition): r for r in todo}
        done, failed = 0, 0
        start = time.time()
        for future in as_completed(futures):
            record = futures[future]
            try:
                result = future.result()
                results[result["user_id"]] = result
            except Exception as e:
                failed += 1
                print(f"FAILED user_id={record['user_id']}: {type(e).__name__}: {e}")
            done += 1
            if done % checkpoint_every == 0 or done == len(todo):
                with open(save_path, "w") as f:
                    json.dump(results, f, indent=2)
                elapsed = time.time() - start
                print(f"  {done}/{len(todo)} processed ({failed} failed), {elapsed:.0f}s elapsed, checkpointed")

    with open(save_path, "w") as f:
        json.dump(results, f, indent=2)
    return results

## ALS: paired sampling

For each domain, sample users who have valid recs from **both** the
multi-domain (collective) and single-domain ALS models, and enough train
history to build a persona from -- same person, two sets of recommendations.

In [16]:
random.seed(676767)  # reproducible sample


def index_by_user(recs):
    return {r['user_id']: r for r in recs if has_valid_recs(r)}


def sample_paired_users(multi_recs, single_recs, train_df, domain, n=N_USERS_PER_DOMAIN):
    multi_by_user = index_by_user(multi_recs)
    single_by_user = index_by_user(single_recs)
    shared_ids = list(set(multi_by_user) & set(single_by_user))
    random.shuffle(shared_ids)

    train_by_user = {uid: g for uid, g in train_df[train_df['domain'] == domain].groupby('user_id')}

    sampled = []
    skipped_no_history = 0
    for user_id in shared_ids:
        if len(sampled) >= n:
            break
        group = train_by_user.get(user_id)
        if group is None:
            skipped_no_history += 1
            continue
        history = get_valid_history(group)
        if history is None:
            skipped_no_history += 1
            continue
        multi_entry, single_entry = multi_by_user[user_id], single_by_user[user_id]
        sampled.append({
            'user_id': user_id,
            'domain': domain,
            'history': history,
            'multi_recommended_item_ids': multi_entry['recommended_item_ids'],
            'multi_recommended_titles': multi_entry['recommended_titles'],
            'single_recommended_item_ids': single_entry['recommended_item_ids'],
            'single_recommended_titles': single_entry['recommended_titles'],
        })

    print(f"{domain}: sampled {len(sampled)}/{n} requested users "
          f"({len(shared_ids)} had recs from both models, {skipped_no_history} skipped for missing/thin history)")
    return sampled


with open(f'{AB_TEST_DIR}/als_multi_books_recs_bulk.json') as f:
    als_multi_books_recs = json.load(f)
with open(f'{AB_TEST_DIR}/als_single_books_recs_bulk.json') as f:
    als_single_books_recs = json.load(f)
with open(f'{AB_TEST_DIR}/als_multi_movies_recs_bulk.json') as f:
    als_multi_movies_recs = json.load(f)
with open(f'{AB_TEST_DIR}/als_single_movies_recs_bulk.json') as f:
    als_single_movies_recs = json.load(f)

als_sampled_books = sample_paired_users(als_multi_books_recs, als_single_books_recs, books_train, 'books')
als_sampled_movies = sample_paired_users(als_multi_movies_recs, als_single_movies_recs, movies_train, 'movies')

als_sampled_books[0]  # sanity check

books: sampled 250/250 requested users (127188 had recs from both models, 0 skipped for missing/thin history)
movies: sampled 250/250 requested users (27194 had recs from both models, 0 skipped for missing/thin history)


{'user_id': 'AHY6Q66BWBTH62EZDYWXVTWTRCOA',
 'domain': 'books',
 'history': [{'title': 'Understudy for Death (Hard Case Crime)',
   'rating': 3.0},
  {'title': 'The Late Show', 'rating': 4.0},
  {'title': 'A Quiet Place', 'rating': 2.0}],
 'multi_recommended_item_ids': ['books::B01BU1ITMI',
  'books::B01MYDJB6R',
  'books::0316225886',
  'books::B00U6DNZOY',
  'books::B00IJJUIMY',
  'books::0316225908',
  'books::0316225932',
  'books::B06W9NKRXG',
  'books::0804178801',
  'books::B0076DELIG'],
 'multi_recommended_titles': ['The Wrong Side of Goodbye (A Harry Bosch Novel Book 19)',
  'The Late Show (Renee Ballard Book 1)',
  'The Crossing (A Harry Bosch Novel, 18)',
  'The Crossing (A Harry Bosch Novel Book 18)',
  'The Burning Room (A Harry Bosch Novel Book 17)',
  'Two Kinds of Truth (A Harry Bosch Novel, 20)',
  'The Burning Room (A Harry Bosch Novel, 17)',
  'Two Kinds of Truth (A Harry Bosch Novel Book 20)',
  'Night School',
  'The Black Box (A Harry Bosch Novel Book 16)'],
 'sin

In [17]:
als_book_profiles = generate_profiles(als_sampled_books, f'{AB_TEST_DIR}/als_domain_book_profiles.json')
als_movie_profiles = generate_profiles(als_sampled_movies, f'{AB_TEST_DIR}/als_domain_movie_profiles.json')

250 profiles to generate (0 already done)
  10/250 processed (0 failed), 5s elapsed, checkpointed
  20/250 processed (0 failed), 9s elapsed, checkpointed
  30/250 processed (0 failed), 12s elapsed, checkpointed
  40/250 processed (0 failed), 15s elapsed, checkpointed
  50/250 processed (0 failed), 19s elapsed, checkpointed
  60/250 processed (0 failed), 23s elapsed, checkpointed
  70/250 processed (0 failed), 26s elapsed, checkpointed
  80/250 processed (0 failed), 30s elapsed, checkpointed
  90/250 processed (0 failed), 33s elapsed, checkpointed
  100/250 processed (0 failed), 36s elapsed, checkpointed
  110/250 processed (0 failed), 40s elapsed, checkpointed
  120/250 processed (0 failed), 43s elapsed, checkpointed
  130/250 processed (0 failed), 46s elapsed, checkpointed
  140/250 processed (0 failed), 50s elapsed, checkpointed
  150/250 processed (0 failed), 53s elapsed, checkpointed
  160/250 processed (0 failed), 56s elapsed, checkpointed
  170/250 processed (0 failed), 60s elaps

## ALS: evaluating both conditions

In [18]:
als_book_multi_evals = evaluate_users(list(als_book_profiles.values()), 'multi', f'{AB_TEST_DIR}/als_domain_book_multi_evaluations.json')
als_book_single_evals = evaluate_users(list(als_book_profiles.values()), 'single', f'{AB_TEST_DIR}/als_domain_book_single_evaluations.json')
als_movie_multi_evals = evaluate_users(list(als_movie_profiles.values()), 'multi', f'{AB_TEST_DIR}/als_domain_movie_multi_evaluations.json')
als_movie_single_evals = evaluate_users(list(als_movie_profiles.values()), 'single', f'{AB_TEST_DIR}/als_domain_movie_single_evaluations.json')

250 'multi' evaluations to run (0 already done)
  10/250 processed (0 failed), 10s elapsed, checkpointed
  20/250 processed (0 failed), 15s elapsed, checkpointed
  30/250 processed (0 failed), 20s elapsed, checkpointed
  40/250 processed (0 failed), 29s elapsed, checkpointed
  50/250 processed (0 failed), 34s elapsed, checkpointed
  60/250 processed (0 failed), 41s elapsed, checkpointed
  70/250 processed (0 failed), 47s elapsed, checkpointed
  80/250 processed (0 failed), 54s elapsed, checkpointed
  90/250 processed (0 failed), 60s elapsed, checkpointed
  100/250 processed (0 failed), 66s elapsed, checkpointed
  110/250 processed (0 failed), 74s elapsed, checkpointed
  120/250 processed (0 failed), 81s elapsed, checkpointed
  130/250 processed (0 failed), 88s elapsed, checkpointed
  140/250 processed (0 failed), 94s elapsed, checkpointed
  150/250 processed (0 failed), 102s elapsed, checkpointed
  160/250 processed (0 failed), 108s elapsed, checkpointed
  170/250 processed (0 failed),

## ALS: paired comparison (single vs multi)

Each user has a rate under both conditions, so this is a one-sample z-test
on the per-user (multi - single) difference, not two independent samples.

In [19]:
from statsmodels.stats.weightstats import ztest


def user_level_rates(evals_dict, k=10):
    return pd.DataFrame([
        {'user_id': r['user_id'], 'view_rate': r['n_view'] / k, 'cart_rate': r['n_cart'] / k, 'buy_rate': r['n_buy'] / k}
        for r in evals_dict.values()
    ]).set_index('user_id')


def paired_diff(multi_rates, single_rates, metric):
    joined = multi_rates[[metric]].join(single_rates[[metric]], lsuffix='_multi', rsuffix='_single', how='inner')
    return (joined[f'{metric}_multi'] - joined[f'{metric}_single']).reset_index(drop=True)


als_book_multi_rates = user_level_rates(als_book_multi_evals)
als_book_single_rates = user_level_rates(als_book_single_evals)
als_movie_multi_rates = user_level_rates(als_movie_multi_evals)
als_movie_single_rates = user_level_rates(als_movie_single_evals)

rows = []
for metric in ('view_rate', 'cart_rate', 'buy_rate'):
    books_diffs = paired_diff(als_book_multi_rates, als_book_single_rates, metric)
    movies_diffs = paired_diff(als_movie_multi_rates, als_movie_single_rates, metric)
    macro_diffs = pd.concat([books_diffs, movies_diffs], ignore_index=True)

    for domain_name, diffs, multi_r, single_r in [
        ('books', books_diffs, als_book_multi_rates, als_book_single_rates),
        ('movies', movies_diffs, als_movie_multi_rates, als_movie_single_rates),
        ('macro_avg', macro_diffs,
         pd.concat([als_book_multi_rates, als_movie_multi_rates]),
         pd.concat([als_book_single_rates, als_movie_single_rates])),
    ]:
        z_stat, p_z = ztest(diffs, value=0, alternative='two-sided')
        rows.append({
            'domain': domain_name, 'metric': metric,
            'multi_mean': round(multi_r[metric].mean(), 4),
            'single_mean': round(single_r[metric].mean(), 4),
            'mean_diff': round(diffs.mean(), 4),
            'n_pairs': len(diffs),
            'z_stat': round(z_stat, 3), 'z_p': p_z,
        })

als_ablation_results_df = pd.DataFrame(rows)
als_ablation_results_df['significant_at_05'] = als_ablation_results_df['z_p'] < 0.05
als_ablation_results_df.to_csv(f'{AB_TEST_DIR}/als_domain_ablation_results.csv', index=False)
als_ablation_results_df

,domain,metric,multi_mean,single_mean,mean_diff,n_pairs,z_stat,z_p,significant_at_05
0,books,view_rate,0.7420,0.8036,-0.0616,250,-4.026,0.000057,True
1,movies,view_rate,0.9068,0.9308,-0.0237,249,-2.610,0.009059,True
2,macro_avg,view_rate,0.8242,0.8672,-0.0427,499,-4.777,0.000002,True
3,books,cart_rate,0.4904,0.5544,-0.0640,250,-4.220,0.000024,True
4,movies,cart_rate,0.6739,0.6960,-0.0209,249,-1.704,0.088317,False
5,macro_avg,cart_rate,0.5820,0.6252,-0.0425,499,-4.339,0.000014,True
6,books,buy_rate,0.4436,0.4960,-0.0524,250,-3.614,0.000301,True
7,movies,buy_rate,0.6353,0.6468,-0.0100,249,-0.782,0.434340,False
8,macro_avg,buy_rate,0.5393,0.5714,-0.0313,499,-3.215,0.001303,True


## BERT4Rec: paired sampling

Same pattern as the ALS section: sample users who have valid recs from
**both** the joint (multi-domain) and single-domain BERT4Rec models, plus
enough train history for a persona. Requires `train_bert4rec_single_domain`
to return the trained model bundle (`transformer` branch) and the
`bert_multi_*`/`bert_single_*` recs_bulk export cell to have been run
(end of `notebooks/03_sequential.ipynb`, same branch).

In [20]:
with open(f'{AB_TEST_DIR}/bert_multi_books_recs_bulk.json') as f:
    bert_multi_books_recs = json.load(f)
with open(f'{AB_TEST_DIR}/bert_single_books_recs_bulk.json') as f:
    bert_single_books_recs = json.load(f)
with open(f'{AB_TEST_DIR}/bert_multi_movies_recs_bulk.json') as f:
    bert_multi_movies_recs = json.load(f)
with open(f'{AB_TEST_DIR}/bert_single_movies_recs_bulk.json') as f:
    bert_single_movies_recs = json.load(f)

bert_sampled_books = sample_paired_users(bert_multi_books_recs, bert_single_books_recs, books_train, 'books')
bert_sampled_movies = sample_paired_users(bert_multi_movies_recs, bert_single_movies_recs, movies_train, 'movies')

bert_sampled_books[0]  # sanity check

books: sampled 250/250 requested users (126384 had recs from both models, 0 skipped for missing/thin history)
movies: sampled 250/250 requested users (41813 had recs from both models, 0 skipped for missing/thin history)


{'user_id': 'AH4WZPYKRO54XCBCKWTVGXF6753Q',
 'domain': 'books',
 'history': [{'title': 'Thru the Bible Vol. 33: The Prophets (Malachi) (33)',
   'rating': 5.0},
  {'title': "Tender Warrior: Every Man's Purpose, Every Woman's Dream, Every Child's Hope",
   'rating': 5.0},
  {'title': 'The Complete Peanuts 1975-1978, Vol. 13-14', 'rating': 5.0},
  {'title': 'The Complete Peanuts 1950-1954 Box Set', 'rating': 5.0},
  {'title': 'Complete Peanuts 1963-1966', 'rating': 5.0},
  {'title': 'The Complete Peanuts 1959-1962 Box Set', 'rating': 5.0}],
 'multi_recommended_item_ids': ['books::0545128285',
  'books::0545162076',
  'books::0763622281',
  'books::0545265355',
  'books::0545010225',
  'books::0545044251',
  'books::1440503257',
  'books::0375851569',
  'books::0740748475',
  'books::0439785960'],
 'multi_recommended_titles': ['The Tales of Beedle the Bard, Standard Edition (Harry Potter)',
  'Harry Potter Paperback Box Set (Books 1-7)',
  'Encyclopedia Prehistorica Dinosaurs : The Defini

In [21]:
bert_book_profiles = generate_profiles(bert_sampled_books, f'{AB_TEST_DIR}/bert_domain_book_profiles.json')
bert_movie_profiles = generate_profiles(bert_sampled_movies, f'{AB_TEST_DIR}/bert_domain_movie_profiles.json')

250 profiles to generate (0 already done)
  10/250 processed (0 failed), 5s elapsed, checkpointed
  20/250 processed (0 failed), 8s elapsed, checkpointed
  30/250 processed (0 failed), 11s elapsed, checkpointed
  40/250 processed (0 failed), 14s elapsed, checkpointed
  50/250 processed (0 failed), 17s elapsed, checkpointed
  60/250 processed (0 failed), 20s elapsed, checkpointed
  70/250 processed (0 failed), 23s elapsed, checkpointed
  80/250 processed (0 failed), 27s elapsed, checkpointed
  90/250 processed (0 failed), 30s elapsed, checkpointed
  100/250 processed (0 failed), 33s elapsed, checkpointed
  110/250 processed (0 failed), 37s elapsed, checkpointed
  120/250 processed (0 failed), 40s elapsed, checkpointed
  130/250 processed (0 failed), 43s elapsed, checkpointed
  140/250 processed (0 failed), 46s elapsed, checkpointed
  150/250 processed (0 failed), 49s elapsed, checkpointed
  160/250 processed (0 failed), 52s elapsed, checkpointed
  170/250 processed (0 failed), 56s elaps

## BERT4Rec: evaluating both conditions

In [22]:
bert_book_multi_evals = evaluate_users(list(bert_book_profiles.values()), 'multi', f'{AB_TEST_DIR}/bert_domain_book_multi_evaluations.json')
bert_book_single_evals = evaluate_users(list(bert_book_profiles.values()), 'single', f'{AB_TEST_DIR}/bert_domain_book_single_evaluations.json')
bert_movie_multi_evals = evaluate_users(list(bert_movie_profiles.values()), 'multi', f'{AB_TEST_DIR}/bert_domain_movie_multi_evaluations.json')
bert_movie_single_evals = evaluate_users(list(bert_movie_profiles.values()), 'single', f'{AB_TEST_DIR}/bert_domain_movie_single_evaluations.json')

250 'multi' evaluations to run (0 already done)
  10/250 processed (0 failed), 10s elapsed, checkpointed
  20/250 processed (0 failed), 15s elapsed, checkpointed
  30/250 processed (0 failed), 22s elapsed, checkpointed
  40/250 processed (0 failed), 28s elapsed, checkpointed
  50/250 processed (0 failed), 35s elapsed, checkpointed
  60/250 processed (0 failed), 42s elapsed, checkpointed
  70/250 processed (0 failed), 49s elapsed, checkpointed
  80/250 processed (0 failed), 55s elapsed, checkpointed
  90/250 processed (0 failed), 62s elapsed, checkpointed
  100/250 processed (0 failed), 68s elapsed, checkpointed
  110/250 processed (0 failed), 75s elapsed, checkpointed
  120/250 processed (0 failed), 81s elapsed, checkpointed
  130/250 processed (0 failed), 87s elapsed, checkpointed
  140/250 processed (0 failed), 93s elapsed, checkpointed
  150/250 processed (0 failed), 100s elapsed, checkpointed
  160/250 processed (0 failed), 106s elapsed, checkpointed
  170/250 processed (0 failed),

## BERT4Rec: paired comparison (single vs multi)

Same one-sample z-test on the per-user (multi - single) difference as the
ALS section.

In [23]:
bert_book_multi_rates = user_level_rates(bert_book_multi_evals)
bert_book_single_rates = user_level_rates(bert_book_single_evals)
bert_movie_multi_rates = user_level_rates(bert_movie_multi_evals)
bert_movie_single_rates = user_level_rates(bert_movie_single_evals)

rows = []
for metric in ('view_rate', 'cart_rate', 'buy_rate'):
    books_diffs = paired_diff(bert_book_multi_rates, bert_book_single_rates, metric)
    movies_diffs = paired_diff(bert_movie_multi_rates, bert_movie_single_rates, metric)
    macro_diffs = pd.concat([books_diffs, movies_diffs], ignore_index=True)

    for domain_name, diffs, multi_r, single_r in [
        ('books', books_diffs, bert_book_multi_rates, bert_book_single_rates),
        ('movies', movies_diffs, bert_movie_multi_rates, bert_movie_single_rates),
        ('macro_avg', macro_diffs,
         pd.concat([bert_book_multi_rates, bert_movie_multi_rates]),
         pd.concat([bert_book_single_rates, bert_movie_single_rates])),
    ]:
        z_stat, p_z = ztest(diffs, value=0, alternative='two-sided')
        rows.append({
            'domain': domain_name, 'metric': metric,
            'multi_mean': round(multi_r[metric].mean(), 4),
            'single_mean': round(single_r[metric].mean(), 4),
            'mean_diff': round(diffs.mean(), 4),
            'n_pairs': len(diffs),
            'z_stat': round(z_stat, 3), 'z_p': p_z,
        })

bert_ablation_results_df = pd.DataFrame(rows)
bert_ablation_results_df['significant_at_05'] = bert_ablation_results_df['z_p'] < 0.05
bert_ablation_results_df.to_csv(f'{AB_TEST_DIR}/bert_domain_ablation_results.csv', index=False)
bert_ablation_results_df

,domain,metric,multi_mean,single_mean,mean_diff,n_pairs,z_stat,z_p,significant_at_05
0,books,view_rate,0.7040,0.7616,-0.0576,250,-3.281,0.001036,True
1,movies,view_rate,0.8344,0.8628,-0.0284,250,-2.338,0.019408,True
2,macro_avg,view_rate,0.7692,0.8122,-0.0430,500,-4.024,0.000057,True
3,books,cart_rate,0.4048,0.4700,-0.0652,250,-4.227,0.000024,True
4,movies,cart_rate,0.5444,0.5744,-0.0300,250,-1.964,0.049535,True
5,macro_avg,cart_rate,0.4746,0.5222,-0.0476,500,-4.379,0.000012,True
6,books,buy_rate,0.3564,0.4056,-0.0492,250,-3.528,0.000418,True
7,movies,buy_rate,0.4976,0.5156,-0.0180,250,-1.265,0.205964,False
8,macro_avg,buy_rate,0.4270,0.4606,-0.0336,500,-3.368,0.000758,True


## Comparing the ALS and BERT4Rec ablations

Both models' single-vs-multi effect sizes side by side -- same domains/metrics,
so `mean_diff` is directly comparable between the two model families.

In [24]:
als_ablation_results_df['model'] = 'als'
bert_ablation_results_df['model'] = 'bert4rec'

combined_ablation_df = pd.concat([als_ablation_results_df, bert_ablation_results_df], ignore_index=True)
combined_ablation_df = combined_ablation_df[['model', 'domain', 'metric', 'multi_mean', 'single_mean', 'mean_diff', 'n_pairs', 'z_stat', 'z_p', 'significant_at_05']]
combined_ablation_df.to_csv(f'{AB_TEST_DIR}/domain_ablation_combined_results.csv', index=False)
combined_ablation_df

,model,domain,metric,multi_mean,single_mean,mean_diff,n_pairs,z_stat,z_p,significant_at_05
0,als,books,view_rate,0.7420,0.8036,-0.0616,250,-4.026,0.000057,True
1,als,movies,view_rate,0.9068,0.9308,-0.0237,249,-2.610,0.009059,True
2,als,macro_avg,view_rate,0.8242,0.8672,-0.0427,499,-4.777,0.000002,True
3,als,books,cart_rate,0.4904,0.5544,-0.0640,250,-4.220,0.000024,True
4,als,movies,cart_rate,0.6739,0.6960,-0.0209,249,-1.704,0.088317,False
5,als,macro_avg,cart_rate,0.5820,0.6252,-0.0425,499,-4.339,0.000014,True
6,als,books,buy_rate,0.4436,0.4960,-0.0524,250,-3.614,0.000301,True
7,als,movies,buy_rate,0.6353,0.6468,-0.0100,249,-0.782,0.434340,False
8,als,macro_avg,buy_rate,0.5393,0.5714,-0.0313,499,-3.215,0.001303,True
9,bert4rec,books,view_rate,0.7040,0.7616,-0.0576,250,-3.281,0.001036,True
